## 4. Knowledge Distillation (DRONA-KD)
Using a larger teacher model (e.g. `gpt2-medium` 345M) and student (`astra_mini_80m` 80M):
To stay safely within the 15GB VRAM limit of a T4 GPU, we run with a batch size of 2 and gradient accumulation of 16 (effective batch size 32) using 8-bit quantization on the teacher.

## 1. Setup Environment
First, we clone the repository and install all required dependencies.

In [ ]:
# Clone repository
!git clone https://github.com/divyang4481/ASTRA-LM.git
%cd ASTRA-LM

# Install package dependencies
!pip install -e .

# Install bitsandbytes and accelerate for quantization (highly recommended for large models)
!pip install bitsandbytes accelerate

## 2. Prepare Dataset
We can download and tokenize a subset of the **FineWeb-Edu** dataset from Hugging Face. Since cloud servers have fast internet connections, we can extract 10M or 100M tokens quickly.

In [ ]:
# Tokenize and build FineWeb dataset
!export PYTHONPATH=src && python scripts/prepare_gpt2_pretrain_data.py \
    --dataset HuggingFaceFW/fineweb-edu \
    --name sample-10BT \
    --tokenizer gpt2 \
    --train_tokens 10000000 \
    --val_tokens 500000 \
    --out_dir data/fineweb_edu_gpt2_10m

## 3. Pretraining (ASTRA with CHAKRA Attention)
On a T4 (15GB/16GB VRAM), we can increase the batch size from 1 to 8 or 16. Let's run a pretraining run.

In [ ]:
# Run pretraining on CUDA GPU
!export PYTHONPATH=src && python scripts/train.py \
    --model_config configs/model/astra_nano_6gb.yaml \
    --train_config configs/train/laptop_6gb_10m.yaml \
    --data_dir data/fineweb_edu_gpt2_10m \
    --device cuda

## 4. Knowledge Distillation (DRONA-KD)
Using a larger teacher model (e.g. `gpt2-medium` 345M) and student (`astra_mini_80m` 80M):
To stay safely within the 15GB VRAM limit of a T4 GPU, we run with a batch size of 2 and gradient accumulation of 16 (effective batch size 32) using 8-bit quantization on the teacher.

In [ ]:
# Run distillation with 8-bit teacher quantization
!export PYTHONPATH=src && python scripts/train_kd.py \
    --student_config configs/model/astra_mini_80m.yaml \
    --teacher_config gpt2-medium \
    --train_config configs/train/laptop_distill.yaml \
    --data_dir data/fineweb_edu_gpt2_10m \
    --alpha 0.5 \
    --temperature 2.0 \
    --allow_random_teacher \
    --teacher_dtype 8bit \
    --batch_size 2 \
    --grad_accum 16

## 5. Generate Text
Generate text from a saved model checkpoint:

In [ ]:
!export PYTHONPATH=src && python scripts/generate.py \
    --checkpoint outputs/astra_mini_distill/checkpoint-1000.pt \
    --model_config configs/model/astra_mini_80m.yaml \
    --prompt "Deep learning is"